# Xem các đoạn trước / sau repair

Chạy các ô từ trên xuống. Mỗi hàng là một `repair_event`: **đỏ = trước sửa**, **xanh = sau sửa**.

Dùng tokenizer của model đã generate; chỉ nạp tokenizer, không nạp weights. Nếu thiếu thư viện, chạy `%pip install transformers ipython` trong một ô riêng.

Các đoạn thể hiện lịch sử tại thời điểm repair; lần repair sau có thể sửa lại chúng. Đoạn trước sửa gồm cả token bị từ chối, nên độ dài hai bên có thể khác nhau.

In [97]:
RESULT_PATH = "/Users/rcyuh/Downloads/train_DeepMath-103K_beyond_leakage (2).jsonl"
TOKENIZER_PATH = "Qwen/Qwen3-4B"  # Hoặc đường dẫn model trên máy chạy.
SAMPLE_INDEX = 0  # None: sample đầu tiên có repair; đây là sample_index, không phải số dòng.

In [98]:
def load_sample_index(result_path=RESULT_PATH) -> list:
    """
    Load the sample index from the result file.
    """
    import jsonlines

    sample_index = []
    with jsonlines.open(result_path) as reader:
        for i, obj in enumerate(reader):
            sample_index.append(obj["sample_index"])
    return sample_index

In [99]:
sample_index_list = load_sample_index(result_path=RESULT_PATH)

In [100]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)


In [101]:
"""Show before/after repair spans from a v3 JSONL file in Jupyter."""

import json
from html import escape


def show_repairs(path, tokenizer, sample_index=None):
    """Show one sample; default to the first sample with repair events.

    Use the generation tokenizer. Each row is an event at repair time;
    later repairs may overwrite it. The before span includes the rejected
    candidate token, so before/after spans can have different lengths.
    """
    from IPython.display import HTML, display

    with open(path, encoding="utf-8") as source:
        for line in source:
            sample = json.loads(line)
            if (sample_index is None and sample.get("repair_events")) or (
                sample_index is not None and sample["sample_index"] == sample_index
            ):
                break
        else:
            raise ValueError("Không tìm thấy sample phù hợp.")

    def cell(ids):
        text = tokenizer.decode(ids, skip_special_tokens=False)
        return '<pre style="white-space:pre-wrap;margin:0">' + escape(text) + '</pre>'

    rows = []
    for number, event in enumerate(sample.get("repair_events", []), 1):
        rows.append(
            f'<tr><td>{number}</td><td>{event["start_index"]}</td>'
            f'<td style="color:#b91c1c">{cell(event["removed_token_ids"])}</td>'
            f'<td style="color:#15803d">{cell(event["replacement_token_ids"])}</td></tr>'
        )

    title = f'<b>Sample {escape(str(sample["sample_index"]))} — {len(rows)} repairs</b>'
    display(HTML(
        title + '<table style="width:100%;text-align:left;table-layout:fixed">'
        '<thead><tr><th style="width:5%">#</th><th style="width:10%">Token bắt đầu</th>'
        '<th>Trước sửa</th><th>Sau sửa</th></tr></thead>'
        '<tbody>' + ''.join(rows) + '</tbody></table>'
    ))


In [143]:
import random
SAMPLE_INDEX = random.choice(sample_index_list)  # Randomly select a sample index from the list
show_repairs(path=RESULT_PATH, tokenizer=tokenizer, sample_index=SAMPLE_INDEX)

#,Token bắt đầu,Trước sửa,Sau sửa
1,30,"through (iv), and the","through (iv), and I need to check each one"
2,38,check each one. The ground,check each one. Let me start by recalling some
3,1320,? But the problem says the,? But let me check another example. Let me
4,2709,). But the problem says the,"). But the problem says ""which of the following is"
5,2718,"following is equal"", and the","following is equal"", so maybe more than one?"
6,2725,than one? But the given,"than one? But the options are (i), ("
7,2741,(iv). But the given,"(iv). Wait, but the original problem says"
8,2748,"original problem says ""the ground","original problem says ""Which of the following is equal"
9,2761,"²B²)"" and the","²B²)"" and gives four options. But in my"
10,5292,", the user says that the",", in the problem statement, maybe only one is"
